In [0]:
%sql
USE control_produccion_pozos;

-- EL MERGE IDEMPOTENTE DE LA CAPA ORO (Corregido sin la columna faltante)
MERGE INTO alertas_produccion_oro AS destino
USING (
    SELECT 
        -- Generamos un ID único para la alerta concatenando el pozo y el timestamp de forma limpia
        CONCAT(id_pozo, '_', DATE_FORMAT(fecha_hora, 'yyyyMMddHHmmss')) as id_alerta,
        id_pozo,
        fecha_hora,
        presion_psi,
        caudal_bpd,
        
        -- REGLAS DE NEGOCIO (Basadas en tu EDA: el umbral crítico lo fijamos en 800 PSI)
        CASE 
            WHEN presion_psi <= 800.0 AND caudal_bpd <= 1000.0 THEN 'CAIDA_PRESION_CRITICA'
            ELSE 'OPERACION_NORMAL'
        END as tipo_anomalia,
        
        CASE 
            WHEN presion_psi <= 800.0 AND caudal_bpd <= 1000.0 THEN 'ALTO'
            ELSE 'BAJO'
        END as nivel_riesgo,
        
        CASE 
            WHEN presion_psi <= 800.0 AND caudal_bpd <= 1000.0 THEN TRUE
            ELSE FALSE
        END as requiere_parada_planta
    FROM pozos_plata
    -- Oro solo consume los datos validados y limpios por la capa Silver
    WHERE estado_sensor = 'OK'
) AS nuevos_datos
ON destino.id_pozo = nuevos_datos.id_pozo AND destino.fecha_hora = nuevos_datos.fecha_hora

-- MATCH LÓGICO: Si la lectura ya se había evaluado, actualizamos por seguridad (No duplica)
WHEN MATCHED THEN
  UPDATE SET 
    destino.tipo_anomalia = nuevos_datos.tipo_anomalia,
    destino.nivel_riesgo = nuevos_datos.nivel_riesgo,
    destino.requiere_parada_planta = nuevos_datos.requiere_parada_planta

-- NOT MATCHED: Si es una lectura nueva, la insertamos
WHEN NOT MATCHED THEN
  INSERT (id_alerta, id_pozo, fecha_hora, presion_psi, caudal_bpd, tipo_anomalia, nivel_riesgo, requiere_parada_planta)
  VALUES (nuevos_datos.id_alerta, nuevos_datos.id_pozo, nuevos_datos.fecha_hora, nuevos_datos.presion_psi, nuevos_datos.caudal_bpd, nuevos_datos.tipo_anomalia, nuevos_datos.nivel_riesgo, nuevos_datos.requiere_parada_planta);

In [0]:
%sql
SELECT tipo_anomalia, nivel_riesgo, requiere_parada_planta, COUNT(*) as cantidad
FROM control_produccion_pozos.alertas_produccion_oro
GROUP BY tipo_anomalia, nivel_riesgo, requiere_parada_planta;